# Entropy Scaling Study — Evaluation Notebook

Nancy Hu & Yiyao Zhang

**Pipeline overview:**
1. Install & imports
2. Mount Drive + GitHub setup
3. Load model
4. Load & preprocess datasets (Wiki / Fiction / Code)
5. Cross-entropy evaluator
6. Full eval loop with checkpointing
7. Save & push results to GitHub

---
## 1. Install & Imports

In [1]:
import os
import json
import torch
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU")

/scratch/yh6384/entropy_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU available: True
Device: NVIDIA L4


In [2]:
import os
REPO_DIR = os.path.expanduser("~/entropy-scaling-llm-for-information-theory")
os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

Working dir: /home/yh6384/entropy-scaling-llm-for-information-theory


---
## 3. Load Model

In [3]:
import os
os.environ["HF_HOME"] = os.path.expandvars("$SCRATCH/hf_cache")

MODELS = ["Qwen/Qwen3-8B"]

for MODEL_NAME in MODELS:
  MODEL_ID = MODEL_NAME.split("/")[-1]

  tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
  model = AutoModelForCausalLM.from_pretrained(
      MODEL_NAME,
      torch_dtype=torch.bfloat16,
      device_map="auto"
  )
  model.eval()

  DEVICE = next(model.parameters()).device
  print(f"Model loaded: {MODEL_NAME}")
  print(f"Device: {DEVICE}")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [02:36<00:00, 31.24s/it]

Model loaded: Qwen/Qwen3-8B
Device: cuda:0


In [4]:
# Sanity check — forward pass
test_text = "Information theory is the study of"
inputs = tokenizer(test_text, return_tensors="pt")
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

with torch.no_grad():
    out = model(**inputs)
    logits = out.logits[0, -1, :]
    top5 = torch.topk(torch.softmax(logits, dim=-1), 5)

print("Output shape:", out.logits.shape)
print("Top 5 predicted next tokens:")
for score, idx in zip(top5.values, top5.indices):
    print(f"  '{tokenizer.decode(idx)}'  {score.item():.4f}")

Output shape: torch.Size([1, 6, 151936])
Top 5 predicted next tokens:
  ' the'  0.6133
  ' how'  0.1377
  ' information'  0.1069
  ' data'  0.0270
  ' coding'  0.0164


---
## 4. Load & Preprocess Datasets

| Domain | Dataset | HF path |
|--------|---------|----------|
| Encyclopedic | WikiText-103 (article-level) | `Salesforce/wikitext` |
| Narrative fiction | PG-19 (replaces BookCorpus) | `deepmind/pg19` |
| Source code | The Stack Smol — Python | `bigcode/the-stack-smol` |

We sample **200 sequences per domain**,  then tokenize once and cache.

In [5]:
N_SAMPLES        = 200      # sequences per domain
MIN_TOKENS       = 5000     # min length per sequence — based on diagnostics
                            # gives plenty of room for sliding windows at N=2048
CONTEXT_LENGTHS  = [16, 64, 256, 512, 1024, 2048]
STRIDE           = 64       # step between window positions
MAX_POSITIONS    = 20       # cap positions per (sequence, N) — keeps cost bounded

SAMPLE_DIR = os.path.join(REPO_DIR, "data")
os.makedirs(SAMPLE_DIR, exist_ok=True)


In [6]:
# === DIAGNOSTIC: WikiText-103 article length distribution ===
# Concatenates ROWS WITHIN AN ARTICLE using the real "= Title =" header.
# Reports length distribution to inform MIN_TOKENS / N_SAMPLES choice.

import re
ARTICLE_HEADER = re.compile(r"^\s*=\s[^=].*[^=]\s=\s*$")  # matches "= Title =" only

wiki_ds_check = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")

articles_text  = []
current_chunks = []
for item in wiki_ds_check:
    text = item["text"].strip()
    if not text:
        continue
    if ARTICLE_HEADER.match(text):
        if current_chunks:
            articles_text.append(" ".join(current_chunks))
        current_chunks = [text]   # include header
    else:
        current_chunks.append(text)
if current_chunks:
    articles_text.append(" ".join(current_chunks))

# Tokenize in batches
article_lengths = []
BATCH = 256
for i in range(0, len(articles_text), BATCH):
    batch = articles_text[i : i + BATCH]
    encoded = tokenizer(batch, truncation=False, padding=False)["input_ids"]
    article_lengths.extend(len(ids) for ids in encoded)

article_lengths_sorted = sorted(article_lengths, reverse=True)

print(f"Real articles detected:  {len(article_lengths_sorted):,}")
print(f"Top {N_SAMPLES} cutoff:           {article_lengths_sorted[N_SAMPLES-1]:,} tokens")
print(f"Articles >= {MIN_TOKENS:,}:    {sum(1 for L in article_lengths_sorted if L >= MIN_TOKENS):,}")


Real articles detected:  29,436
Top 200 cutoff:           17,527 tokens
Articles >= 5,000:    8,327


In [7]:
# --- Domain 1: Wiki (article-level concatenation, random sampled) ---
# One article = one sequence. Filter by MIN_TOKENS, random sample N_SAMPLES.

wiki_path = os.path.join(SAMPLE_DIR, f"wiki_{MODEL_ID}_tokens.json")

if os.path.exists(wiki_path):
    print(f"[wiki] Already cached — loading.")
    with open(wiki_path) as f:
        wiki_tokens = json.load(f)
else:
    import re
    import random
    ARTICLE_HEADER = re.compile(r"^\s*=\s[^=].*[^=]\s=\s*$")

    wiki_ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")

    # Pass 1: group rows into articles
    articles_text = []
    current_chunks = []
    for item in wiki_ds:
        text = item["text"].strip()
        if not text:
            continue
        if ARTICLE_HEADER.match(text):
            if current_chunks:
                articles_text.append(" ".join(current_chunks))
            current_chunks = [text]
        else:
            current_chunks.append(text)
    if current_chunks:
        articles_text.append(" ".join(current_chunks))

    # Pass 2: tokenize all articles
    print(f"[wiki] Tokenizing {len(articles_text):,} articles...")
    tokenized = []
    BATCH = 256
    for i in range(0, len(articles_text), BATCH):
        batch = articles_text[i : i + BATCH]
        encoded = tokenizer(batch, truncation=False, padding=False)["input_ids"]
        tokenized.extend(encoded)

    # Filter by length, then random sample
    long_enough = [ids for ids in tokenized if len(ids) >= MIN_TOKENS]
    print(f"[wiki] {len(long_enough):,} articles meet MIN_TOKENS={MIN_TOKENS:,}")

    random.seed(42)
    wiki_tokens = random.sample(long_enough, N_SAMPLES)

    with open(wiki_path, "w") as f:
        json.dump(wiki_tokens, f)
    print(f"[wiki] Saved to {wiki_path}")

print(f"Wiki sequences: {len(wiki_tokens)}")
print(f"  shortest: {min(len(s) for s in wiki_tokens):,} tokens")
print(f"  longest:  {max(len(s) for s in wiki_tokens):,} tokens")

[wiki] Already cached — loading.
Wiki sequences: 200
  shortest: 5,007 tokens
  longest:  23,220 tokens


In [8]:
# --- Domain 2: Narrative fiction (Gutenberg English) ---
# One book passage = one sequence. Filter to MIN_TOKENS, keep top N_SAMPLES longest.

fiction_path = os.path.join(SAMPLE_DIR, f"fiction_{MODEL_ID}_tokens.json")

if os.path.exists(fiction_path):
    print(f"[fiction] Already cached — loading.")
    with open(fiction_path) as f:
        fiction_tokens = json.load(f)
else:
    pg19_ds = load_dataset("sedthh/gutenberg_english", split="train", streaming=True)

    fiction_tokens = []
    seen_long_enough = 0
    print(f"[fiction] Sampling — looking for sequences >= {MIN_TOKENS:,} tokens...")
    for item in pg19_ds:
        text = item["TEXT"].strip()
        # cheap pre-filter
        if len(text.split()) < int(MIN_TOKENS * 0.7):
            continue
        token_ids = tokenizer(text, truncation=False, return_tensors=None)["input_ids"]
        if len(token_ids) >= MIN_TOKENS:
            fiction_tokens.append(token_ids)
            seen_long_enough += 1
        if seen_long_enough >= N_SAMPLES:
            break

    with open(fiction_path, "w") as f:
        json.dump(fiction_tokens, f)
    print(f"[fiction] Saved to {fiction_path}")

print(f"Fiction sequences: {len(fiction_tokens)}")
print(f"  shortest: {min(len(s) for s in fiction_tokens):,} tokens")
print(f"  longest:  {max(len(s) for s in fiction_tokens):,} tokens")


[fiction] Already cached — loading.
Fiction sequences: 200
  shortest: 5,918 tokens
  longest:  1,566,307 tokens


In [9]:
# --- Domain 3: Source code (Python alpaca instructions) ---
# Each individual code snippet is short, so we concatenate snippets until
# each sequence is >= MIN_TOKENS. One sequence = one concatenated code passage.

code_path = os.path.join(SAMPLE_DIR, f"code_{MODEL_ID}_tokens.json")

if os.path.exists(code_path):
    print(f"[code] Already cached — loading.")
    with open(code_path) as f:
        code_tokens = json.load(f)
else:
    code_ds = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")

    code_tokens = []
    current_ids = []
    print(f"[code] Sampling — concatenating snippets until each seq >= {MIN_TOKENS:,} tokens...")
    for item in code_ds:
        text = (item["instruction"] + "\n" + item["output"]).strip()
        chunk_ids = tokenizer(text, truncation=False, return_tensors=None)["input_ids"]
        current_ids.extend(chunk_ids)
        if len(current_ids) >= MIN_TOKENS:
            code_tokens.append(current_ids)
            current_ids = []
        if len(code_tokens) >= N_SAMPLES:
            break

    with open(code_path, "w") as f:
        json.dump(code_tokens, f)
    print(f"[code] Saved to {code_path}")

print(f"Code sequences: {len(code_tokens)}")
print(f"  shortest: {min(len(s) for s in code_tokens):,} tokens")
print(f"  longest:  {max(len(s) for s in code_tokens):,} tokens")


[code] Sampling — concatenating snippets until each seq >= 5,000 tokens...
[code] Saved to /home/yh6384/entropy-scaling-llm-for-information-theory/data/code_Qwen3-8B_tokens.json
Code sequences: 200
  shortest: 5,002 tokens
  longest:  8,417 tokens


In [10]:
# Summary
print(f"Wiki     sequences: {len(wiki_tokens):>3}  | shortest: {min(len(s) for s in wiki_tokens):>6,} | longest: {max(len(s) for s in wiki_tokens):>6,}")
print(f"Fiction  sequences: {len(fiction_tokens):>3}  | shortest: {min(len(s) for s in fiction_tokens):>6,} | longest: {max(len(s) for s in fiction_tokens):>6,}")
print(f"Code     sequences: {len(code_tokens):>3}  | shortest: {min(len(s) for s in code_tokens):>6,} | longest: {max(len(s) for s in code_tokens):>6,}")


Wiki     sequences: 200  | shortest:  5,007 | longest: 23,220
Fiction  sequences: 200  | shortest:  5,918 | longest: 1,566,307
Code     sequences: 200  | shortest:  5,002 | longest:  8,417


---
## 5. Cross-Entropy Evaluator

We compute $H_N = -\frac{1}{T} \sum_t \log p(x_t \mid x_{t-N:t-1})$ for each context length $N$.

For each sequence we slide a window of size $N$ and record the negative log-probability of the true next token under the model.

In [11]:
def compute_entropy_at_N(token_ids, N, stride=STRIDE, max_positions=MAX_POSITIONS):
    """
    Cross-entropy of next-token prediction at context length N.

    Slides a window of length N across the sequence with the given stride.
    Caps the number of positions evaluated per sequence to keep cost bounded.
    """
    tokens = torch.tensor(token_ids, dtype=torch.long)
    T = len(tokens)
    if T < N + 1:
        return None

    # All valid positions where we can read N tokens of context and 1 target
    available = list(range(N, T, stride))
    # Cap positions to keep cost equal across N
    positions = available[:max_positions]

    neg_log_probs = []
    for t in positions:
        context = tokens[t - N : t].unsqueeze(0).to(DEVICE)
        target  = tokens[t].item()
        with torch.no_grad():
            out    = model(context)
            logits = out.logits[0, -1, :]
            log_p  = torch.nn.functional.log_softmax(logits, dim=-1)
            neg_log_probs.append(-log_p[target].item())

    return float(np.mean(neg_log_probs)) if neg_log_probs else None


# Unit test
test_h = compute_entropy_at_N(wiki_tokens[0], N=64)
print(f"Unit test H_N=64 on wiki[0]: {test_h:.4f} nats")
assert test_h is not None and test_h > 0
print("Unit test passed.")


Unit test H_N=64 on wiki[0]: 2.0311 nats
Unit test passed.


---
## 6. Full Evaluation Loop

Runs all **18 combinations** (3 domains × 6 context lengths).  
Results are written to `entropy_results.csv` after **every combination** — so a Colab crash loses at most one run.

In [12]:
RESULTS_PATH = os.path.join(REPO_DIR, "data", f"entropy_results_{MODEL_ID}.csv")

if os.path.exists(RESULTS_PATH):
    results_df = pd.read_csv(RESULTS_PATH)
    done = set(zip(results_df["domain"], results_df["N"]))
    print(f"Resuming — {len(results_df)} combinations already done.")
else:
    results_df = pd.DataFrame(columns=["model", "domain", "N", "mean_H", "std_H", "n_sequences", "n_positions_per_seq"])
    done = set()
    print("Starting fresh.")

DOMAINS = {
    "wiki":    wiki_tokens,
    "fiction": fiction_tokens,
    "code":    code_tokens,
}

total_combos = len(DOMAINS) * len(CONTEXT_LENGTHS)
combo_num = 0

for domain_name, token_list in DOMAINS.items():
    for N in CONTEXT_LENGTHS:
        combo_num += 1
        if (domain_name, N) in done:
            print(f"[{combo_num}/{total_combos}] Skipping {domain_name} N={N} (already done)")
            continue

        print(f"[{combo_num}/{total_combos}] {domain_name}  N={N} ...", end=" ", flush=True)

        entropies = []
        for token_ids in token_list:
            h = compute_entropy_at_N(token_ids, N)
            if h is not None:
                entropies.append(h)

        mean_h = float(np.mean(entropies))
        std_h  = float(np.std(entropies))
        n_seq  = len(entropies)

        # Compute typical positions/seq for diagnostics
        sample_seq_len = len(token_list[0])
        n_positions = min(MAX_POSITIONS, max(0, (sample_seq_len - N) // STRIDE))

        print(f"H = {mean_h:.4f} ± {std_h:.4f}  (n_seq={n_seq}, pos/seq={n_positions})")

        new_row = pd.DataFrame([{
            "model": MODEL_ID,
            "domain": domain_name,
            "N": N,
            "mean_H": mean_h,
            "std_H": std_h,
            "n_sequences": n_seq,
            "n_positions_per_seq": n_positions,
        }])
        results_df = pd.concat([results_df, new_row], ignore_index=True)
        results_df.to_csv(RESULTS_PATH, index=False)
        done.add((domain_name, N))

print("\nAll done! Results saved to:", RESULTS_PATH)

# Free GPU memory
del model, tokenizer
import gc
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed after {MODEL_ID}.")


Resuming — 12 combinations already done.
[1/18] Skipping wiki N=16 (already done)
[2/18] Skipping wiki N=64 (already done)
[3/18] Skipping wiki N=256 (already done)
[4/18] Skipping wiki N=512 (already done)
[5/18] Skipping wiki N=1024 (already done)
[6/18] Skipping wiki N=2048 (already done)
[7/18] Skipping fiction N=16 (already done)
[8/18] Skipping fiction N=64 (already done)
[9/18] Skipping fiction N=256 (already done)
[10/18] Skipping fiction N=512 (already done)
[11/18] Skipping fiction N=1024 (already done)
[12/18] Skipping fiction N=2048 (already done)
[13/18] code  N=16 ... H = 1.7921 ± 0.7245  (n_seq=200, pos/seq=20)
[14/18] code  N=64 ... H = 1.3045 ± 0.6629  (n_seq=200, pos/seq=20)
[15/18] code  N=256 ... H = 1.0305 ± 0.5291  (n_seq=200, pos/seq=20)
[16/18] code  N=512 ... H = 0.9637 ± 0.5132  (n_seq=200, pos/seq=20)
[17/18] code  N=1024 ... H = 0.9060 ± 0.5190  (n_seq=200, pos/seq=20)
[18/18] code  N=2048 ... H = 0.8808 ± 0.5213  (n_seq=200, pos/seq=20)

All done! Results s

In [13]:
# Combine the results (after we have run all the models)
import glob
all_csvs = glob.glob(os.path.join(SAMPLE_DIR, "entropy_results_*.csv"))
combined_df = pd.concat([pd.read_csv(f) for f in all_csvs], ignore_index=True)
combined_df.to_csv(os.path.join(SAMPLE_DIR, "entropy_results_combined.csv"), index=False)
print(combined_df.to_string(index=False))

 domain    N   mean_H    std_H  n_sequences           model  n_positions_per_seq
   wiki   16 3.335649 0.761050          200   Llama-2-7b-hf                  NaN
   wiki   64 2.330842 0.692556          200   Llama-2-7b-hf                  NaN
   wiki  256 2.044455 0.680878          200   Llama-2-7b-hf                  NaN
   wiki  512 2.007711 0.709705          200   Llama-2-7b-hf                  NaN
   wiki 1024 1.900979 0.708620          200   Llama-2-7b-hf                  NaN
   wiki 2048 1.561157 2.134707          200   Llama-2-7b-hf                  NaN
fiction   16 3.022961 0.901260          200   Llama-2-7b-hf                  NaN
fiction   64 1.577939 0.664395          200   Llama-2-7b-hf                  NaN
fiction  256 1.348504 0.710967          200   Llama-2-7b-hf                  NaN
fiction  512 1.349141 0.687810          200   Llama-2-7b-hf                  NaN
fiction 1024 1.398988 0.732667          200   Llama-2-7b-hf                  NaN
fiction 2048 1.368932 2.2539

In [14]:
# Preview results table
results_df = pd.read_csv(RESULTS_PATH)
print(results_df.to_string(index=False))

   model  domain    N   mean_H    std_H  n_sequences  n_positions_per_seq
Qwen3-8B    wiki   16 3.178024 0.753680          200                   20
Qwen3-8B    wiki   64 2.503610 0.827488          200                   20
Qwen3-8B    wiki  256 2.370478 0.757529          200                   20
Qwen3-8B    wiki  512 2.346419 0.736519          200                   20
Qwen3-8B    wiki 1024 2.321117 0.705774          200                   20
Qwen3-8B    wiki 2048 2.184678 0.679121          200                   20
Qwen3-8B fiction   16 3.790733 0.985650          200                   20
Qwen3-8B fiction   64 2.685759 0.859949          200                   20
Qwen3-8B fiction  256 2.543787 0.864213          200                   20
Qwen3-8B fiction  512 2.566393 0.867246          200                   20
Qwen3-8B fiction 1024 2.583298 0.943926          200                   20
Qwen3-8B fiction 2048 2.486822 0.859909          200                   20
Qwen3-8B    code   16 1.792113 0.72449

---
## 7. Push Results to GitHub

In [15]:
# Push results back to GitHub
os.chdir(REPO_DIR)

!git add data/
!git status
!git commit -m "add Qwen3-8B eval results"
!git push origin main

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   data/entropy_results_Qwen3-8B.csv
	new file:   data/entropy_results_combined.csv

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   entropy_eval_long_version_hpc_qwen.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.ipynb_checkpoints/entropy_eval_long_version_hpc_qwen-checkpoint.ipynb

[main 623b317] add Qwen3-8B eval results with article-level wiki, MIN_TOKENS=5000
 2 files changed, 79 insertions(+)
 create mode 100644 data/entropy_results_combined.csv
To github.com:yh6384-design/entropy-scaling-llm-for-information-theory.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'github.com:yh6384-design/entropy-scaling-llm-for-information-t

In [17]:
!cd ~/entropy-scaling-llm-for-information-theory
!git pull origin main --rebase
!git push origin main

error: cannot pull with rebase: You have unstaged changes.
error: Please commit or stash them.
To github.com:yh6384-design/entropy-scaling-llm-for-information-theory.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'github.com:yh6384-design/entropy-scaling-llm-for-information-theory.git'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [16]:
# Debug: trace a single code sequence at N=16
test_seq = code_tokens[0]
N = 16
stride = 64
max_positions = 20

import torch
import numpy as np

tokens = torch.tensor(test_seq, dtype=torch.long)
T = len(tokens)
print(f"Sequence length: {T}")
print(f"Min token ID: {tokens.min().item()}, Max: {tokens.max().item()}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Any out-of-range? {(tokens >= tokenizer.vocab_size).any().item()}")

available = list(range(N, T, stride))[:max_positions]
print(f"Will evaluate at {len(available)} positions: {available[:5]}...")

# Try one forward pass
t = available[0]
context = tokens[t - N : t].unsqueeze(0).to(DEVICE)
target = tokens[t].item()
print(f"\nFirst eval: context shape {context.shape}, target token {target}")

with torch.no_grad():
    out = model(context)
    logits = out.logits[0, -1, :]
    print(f"Logits has NaN? {torch.isnan(logits).any().item()}")
    print(f"Logits has Inf? {torch.isinf(logits).any().item()}")
    print(f"Logits range: [{logits.min().item():.2f}, {logits.max().item():.2f}]")
    
    log_p = torch.nn.functional.log_softmax(logits, dim=-1)
    print(f"Log-probs has NaN? {torch.isnan(log_p).any().item()}")
    
    target_logp = log_p[target].item()
    print(f"Target log-prob: {target_logp}")
    print(f"Negative log-prob: {-target_logp}")

Sequence length: 5032
Min token ID: 0, Max: 98223


NameError: name 'tokenizer' is not defined

In [ ]:
# Find which code sequences produce NaN at N=16
import torch
import numpy as np

bad_seqs = []
all_h = []

for i, token_ids in enumerate(code_tokens):
    h = compute_entropy_at_N(token_ids, N=16)
    all_h.append(h)
    if h is None or (isinstance(h, float) and np.isnan(h)):
        bad_seqs.append(i)
        print(f"  Sequence {i}: h = {h}, length = {len(token_ids)}")

print(f"\nTotal sequences: {len(code_tokens)}")
print(f"Bad sequences: {len(bad_seqs)}")
print(f"Good sequences with finite h: {sum(1 for h in all_h if h is not None and not np.isnan(h))}")

if bad_seqs:
    print(f"\nFirst bad sequence index: {bad_seqs[0]}")
    bad = code_tokens[bad_seqs[0]]
    bad_t = torch.tensor(bad, dtype=torch.long)
    print(f"  Length: {len(bad)}")
    print(f"  Min/max token: {bad_t.min().item()}/{bad_t.max().item()}")
    print(f"  First 30 tokens: {bad[:30]}")